In [1]:
#| default_exp lawa

In [2]:
#| hide
import nbdev; nbdev.nbdev_export()

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"


In [4]:
#| export
from os import getenv
seq_length = 128000
model_path = getenv("MODEL")


In [5]:
model_path = 'lawa'

In [6]:
#| export
full_path = f'./models/{model_path}'


In [7]:
#| export
from vllm import LLM, SamplingParams
llm = LLM(model=full_path, dtype='bfloat16', device="cuda", max_seq_len_to_capture=seq_length, gpu_memory_utilization=0.5)
from front.common import process_seq


INFO 09-28 05:37:39 config.py:1652] Downcasting torch.float32 to torch.bfloat16.
WARNING 09-28 05:37:39 arg_utils.py:930] Chunked prefill is enabled by default for models with max_model_len > 32K. Currently, chunked prefill might not work with some features or models. If you encounter any issues, please disable chunked prefill by setting --enable-chunked-prefill=False.
INFO 09-28 05:37:39 config.py:1010] Chunked prefill is enabled with max_num_batched_tokens=512.
INFO 09-28 05:37:39 llm_engine.py:226] Initializing an LLM engine (v0.6.1.dev238+ge2c6e0a82) with config: model='./models/lawa', speculative_config=None, tokenizer='./models/lawa', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, 

[W928 05:37:40.073642956 socket.cpp:697] [c10d] The client socket cannot be initialized to connect to [bbb]:48223 (errno: 97 - Address family not supported by protocol).


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 09-28 05:37:41 model_runner.py:1025] Loading model weights took 2.3185 GB
INFO 09-28 05:37:41 gpu_executor.py:122] # GPU blocks: 16798, # CPU blocks: 8192
INFO 09-28 05:37:42 model_runner.py:1329] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 09-28 05:37:42 model_runner.py:1333] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 09-28 05:37:56 model_runner.py:1456] Graph capturing finished in 14 secs.


In [8]:
#| export
def get_sample(prompt, length: int, num_samples: int, allow_linebreak: bool, temperature: float = 0.5):
    sampling_params = SamplingParams(
        temperature=temperature,
        top_k=10,
        top_p=0.95,
        max_tokens=length,
        n=num_samples,
        repetition_penalty=2,
    )
    
    outputs = llm.generate(prompt, sampling_params=sampling_params)
    result = [oo.text for o in outputs for oo in o.outputs]
    # Post-process the generated sequences
    result = [res.replace('\n', ' ') for res in result]
    result = process_seq(result)
    return result



In [10]:
%%time
get_sample('На словах ты Лев Толстой, а на деле'*1000, 50, 4, False)

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.37it/s, est. speed input: 21980.50 toks/s, output: 274.77 toks/s]

CPU times: user 16.8 s, sys: 36.8 ms, total: 16.8 s
Wall time: 760 ms


[' он был очень занят и быстро.',
 ' он не был ни левой или правым.',
 ' он не был ни лем или правым. А то было еще как – но теперь уже ничего подобного нет… Знаешь что я говорю? Ты ведь все равно никогда о нем даже и думать просто так нельзя!',
 ' он не был ни левой сиськой или правейшей ногами.']